# Мониторинг фин. эффекта по Excel

Оценка без факта ПСР:

```
e_fee = p_fu×100k + p_court×(100k+15k)
expected_psr = precision × Σ_I (paid×k + e_fee)
net = expected_psr − Σ_I paid
```

Интервенция `I`: `РезультатПроверки=1` ∧ `Заключено соглашение=1` ∧ `Выплата по модели=1`.

`PAID_COL` — колонка «в карман» (после explore можно зафиксировать явно). Приоры — из `data/retro_priors.json` или с ретро-parquet.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in (_here, *_here.parents) if (p / "pyproject.toml").exists()
)
SRC = PROJECT_ROOT / "src"
for _p in (SRC, PROJECT_ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

NOTEBOOK_DIR = PROJECT_ROOT / "monitoring" / "fin_effects"
DATA_DIR = NOTEBOOK_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT", PROJECT_ROOT)
print("DATA_DIR", DATA_DIR)

In [ ]:
from IPython.display import display

from querulus.fin_effect.excel_monitoring import (
    default_demo_priors,
    estimate_monitoring_effect,
    load_excel,
    load_retro_priors,
    save_retro_priors,
    sensitivity_table,
    write_synthetic_claims_excel,
)

GENERATE_SYNTHETIC = True
EXCEL_PATH = DATA_DIR / "querulus_claims_synthetic.xlsx"
# EXCEL_PATH = DATA_DIR / "claims_prod.xlsx"

PRIORS_PATH = DATA_DIR / "retro_priors.json"
# После explore зафиксируйте колонку явно, иначе — эвристика (recommended → payment → …)
PAID_COL = None  # например: "Сумма рекомендованная к доплате по модулю"

# Опционально: ретро parquet с TARGET_FREQ / preds_cf для пересчёта priors
RETRO_PARQUET = None  # Path(...) / "df_for_service.parquet"
RETRO_THRESHOLD = 0.5

if GENERATE_SYNTHETIC and not Path(EXCEL_PATH).exists():
    write_synthetic_claims_excel(EXCEL_PATH, n_rows=300)
    print("synthetic written", EXCEL_PATH)

if not PRIORS_PATH.exists():
    save_retro_priors(default_demo_priors(), PRIORS_PATH)
    print("demo priors written", PRIORS_PATH)

df = load_excel(EXCEL_PATH)
print("shape", df.shape)

In [ ]:
if RETRO_PARQUET is not None and Path(RETRO_PARQUET).exists():
    import pandas as pd
    from querulus.fin_effect.excel_monitoring import compute_retro_priors

    retro = pd.read_parquet(RETRO_PARQUET)
    priors = compute_retro_priors(retro, threshold=RETRO_THRESHOLD)
    save_retro_priors(priors, PRIORS_PATH)
    print("priors from parquet →", PRIORS_PATH)
else:
    priors = load_retro_priors(PRIORS_PATH)
    print("priors from json")

print(priors)
print("e_fee", round(priors.expected_fee(), 2))

In [ ]:
effect = estimate_monitoring_effect(df, priors, paid_col=PAID_COL)
summary = {
    "paid_column": effect.paid_column,
    "n_intervention": effect.n_intervention,
    "sum_paid": round(effect.sum_paid, 2),
    "e_fee": round(effect.e_fee, 2),
    "expected_psr": round(effect.expected_psr, 2),
    "cost": round(effect.cost, 2),
    "net": round(effect.net, 2),
}
display(summary)

sens = sensitivity_table(df, priors, paid_col=PAID_COL)
display(sens)